### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

In [4]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain.prompts import PromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

d:\project\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
## step1 : Load and split the dataset
loader = TextLoader("data/langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)


In [6]:
chunks

[Document(metadata={'source': 'data/langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'data/langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'data/langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse 

In [7]:
# step 2: vector store
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(chunks,embedding_model)

#Step 3: MMR Retriever
retriever=vectorstore.as_retriever(search_type="mmr",search_kwargs={"k":5})
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001EB0BE474D0>, search_type='mmr', search_kwargs={'k': 5})

In [8]:
#Step 4: LLM and Prompt

import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm=init_chat_model("groq:llama-3.1-8b-instant")
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001EB0C0C81A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EB0C0C8EC0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [12]:
#Query Expansion 

query_expansion_prompt=PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.
                                                    
Original Query: "{query}"
                        
Expanded Query:
""")

query_expansion_chain=query_expansion_prompt | llm | StrOutputParser()

query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\n\nOriginal Query: "{query}"\n\nExpanded Query:\n')
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001EB0C0C81A0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001EB0C0C8EC0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))
| StrOutputParser()

In [13]:
query_expansion_chain.invoke({"query":"Langchain memory"})

'To improve document retrieval for the query "Langchain memory," I would expand it to include relevant synonyms, technical terms, and useful context. Here\'s an expanded query:\n\n1. **Synonyms and related terms:**\n\t* Langchain core\n\t* Langchain storage\n\t* Langchain database\n\t* Langchain knowledge graph\n\t* Memory management in Langchain\n\t* Langchain cache\n2. **Technical terms:**\n\t* Blockchain-based memory\n\t* Distributed memory\n\t* Decentralized memory\n\t* Interoperable memory\n\t* Scalable memory\n3. **Contextual keywords:**\n\t* AI-assisted memory\n\t* Hybrid memory\n\t* Edge computing\n\t* Cloud computing\n\t* Data storage\n\t* Data management\n\t* Blockchain architecture\n4. **Query modifications:**\n\t* "Langchain memory architecture"\n\t* "Designing a Langchain memory system"\n\t* "Langchain memory scalability"\n\t* "Langchain memory security"\n\t* "Langchain memory interoperability"\n\nExpanded Query:\n"((Langchain core OR Langchain storage OR Langchain databas

In [14]:
#RAG answering prompt

answer_prompt=PromptTemplate.from_template("""
Answer the question based on the context below.
                                        
Context: {context}
                                        
Question: {input}
""")

document_chain=create_stuff_documents_chain(llm=llm,prompt=answer_prompt)

In [17]:
#Full rag pipeline with query expansion

rag_pipeline=(
    RunnableMap({
        "input":lambda x: x["input"],
        "context": lambda x:retriever.invoke(query_expansion_chain.invoke({"query":x["input"]}))
    })
    | document_chain
    | StrOutputParser()
)

In [18]:
#Step 6 : Run query

query={"input":"What types of memory does Langcain support?"}
print(query_expansion_chain.invoke({"query":query["input"]}))
response=rag_pipeline.invoke(query)
print("Answer:",response)

Here's an expanded query to improve document retrieval:

"Langcain memory support: What types of RAM, memory technologies (e.g., DRAM, DDR3, DDR4, DDR5, SRAM, MRAM), and memory interfaces (e.g., DIMM, SO-DIMM, RIMM) does Langcain support? 

Provide information on the following:
- Supported memory architectures (e.g., 32-bit, 64-bit)
- Maximum memory capacity per slot and total system capacity
- Memory speed and frequency support (e.g., 1333 MHz, 1600 MHz, 2400 MHz)
- Compatibility with different memory types (e.g., ECC, non-ECC)
- Any specific requirements or limitations for memory configuration"

This expanded query includes relevant synonyms, technical terms, and useful context to help retrieve more accurate and comprehensive information about Langcain's memory support.
Answer: According to the provided context, LangChain supports the following types of memory modules:

1. ConversationBufferMemory: This module allows the LLM to maintain awareness of previous conversation turns.
2. Co